In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize local SparkSession
spark = (
    SparkSession.builder
    .appName("01_Bronze_Ingestion")
    .getOrCreate()
)

BRONZE_PATH = "/home/jovyan/data_lake/bronze"
Path(BRONZE_PATH).mkdir(parents=True, exist_ok=True)

INGESTION_TS = datetime.now(timezone.utc).isoformat()
COUNTRY_CODE = "EGY"
manifest_records = []

def write_bronze_parquet(df, table_name: str, source_name: str) -> None:
    """Write raw DataFrame to local Parquet storage and log ingestion metadata."""
    enriched_df = (
        df.withColumn("_ingested_at", F.lit(INGESTION_TS))
          .withColumn("_source_system", F.lit(source_name))
    )
    target_path = f"{BRONZE_PATH}/{table_name}"
    enriched_df.write.mode("overwrite").parquet(target_path)

    row_count = enriched_df.count()
    manifest_records.append({
        "source_system": source_name,
        "target_path": target_path,
        "row_count": row_count,
        "status": "SUCCESS",
        "ingested_at": INGESTION_TS,
    })
    print(f"[OK] Loaded {row_count} rows into {target_path}")

# Ingest World Bank API indicators for Egypt
WB_INDICATORS = [
    "EG.ELC.ACCS.ZS",   # Access to electricity (% of population)
    "EG.FEC.RNEW.ZS",   # Renewable energy consumption (% of total final energy)
    "EG.USE.ELEC.KH.PC" # Electric power consumption (kWh per capita)
]

wb_rows = []
for indicator in WB_INDICATORS:
    url = f"https://api.worldbank.org/v2/country/{COUNTRY_CODE}/indicator/{indicator}?format=json&per_page=100"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    if len(payload) > 1 and payload[1]:
        for item in payload[1]:
            wb_rows.append({
                "country_iso3": item.get("countryiso3code"),
                "country_name": item.get("country", {}).get("value"),
                "indicator_id": item.get("indicator", {}).get("id"),
                "indicator_name": item.get("indicator", {}).get("value"),
                "year": item.get("date"),
                "value": item.get("value"),
            })

wb_spark_df = spark.createDataFrame(pd.DataFrame(wb_rows))
write_bronze_parquet(wb_spark_df, "bronze_wb_indicators", "World_Bank_API")

# Persist Bronze Ingestion Manifest
manifest_df = spark.createDataFrame(pd.DataFrame(manifest_records))
manifest_df.write.mode("append").parquet(f"{BRONZE_PATH}/bronze_ingestion_manifest")

print("=== Bronze Ingestion Manifest ===")
spark.read.parquet(f"{BRONZE_PATH}/bronze_ingestion_manifest").show(truncate=False)

spark.stop()